# Simple Earth Engine export notebook

This notebook replaces the older config/export script setup with a much simpler workflow for two use cases:

1. Export SAR images from Earth Engine to Google Drive.
2. Export cloud-free Sentinel-2 images from Earth Engine to Google Drive.

No preprocessing is applied here yet; the goal is only to get data out of Earth Engine.

In [6]:
import json
import os
import pandas as pd
import ee

# Authenticate once if needed.
# ee.Authenticate()

try:
    ee.Initialize(project="war-damage-504215")
    print("Earth Engine initialized.")
except Exception as e:
    print("Initialization failed. Run ee.Authenticate() once and try again.")
    print(e)

Earth Engine initialized.


In [7]:
# Simple project-level config derived from the old config.py idea.
CITY_CONFIG = {
    "Gaza": "2023-10-10",
    "Mosul": "2016-10-16",
    "Raqqa": "2017-06-06",
    "Aleppo": "2016-07-01",
}

def parse_name(csv_path):
    base = os.path.basename(csv_path)
    parts = base.split("_")
    city = parts[0]
    date = parts[1]
    return city, f"{date[:4]}-{date[4:6]}-{date[6:8]}"

In [16]:
import math
import re


def centroid_from_geometry(geom):
    """Return a simple centroid from a footprint geometry."""
    if not isinstance(geom, dict):
        return None

    geom_type = geom.get("type")
    if geom_type == "Point":
        coords = geom.get("coordinates", [])
        if len(coords) >= 2:
            return float(coords[0]), float(coords[1])

    if geom_type == "GeometryCollection":
        for child in geom.get("geometries", []):
            centroid = centroid_from_geometry(child)
            if centroid:
                return centroid

    if geom_type in {"Polygon", "MultiPolygon"}:
        coords = geom.get("coordinates", [])
        points = []
        if geom_type == "Polygon":
            rings = coords
        else:
            rings = [ring for polygon in coords for ring in polygon]

        for ring in rings:
            for point in ring:
                if len(point) >= 2:
                    points.append(point)

        if points:
            lon = sum(p[0] for p in points) / len(points)
            lat = sum(p[1] for p in points) / len(points)
            return float(lon), float(lat)

    return None


def rectangle_around(center_lon, center_lat, size_m=170):
    """Return a size_m x size_m ee.Geometry.Rectangle centered on one point."""
    half_side_m = size_m / 2.0
    meters_per_deg_lat = 111320.0
    meters_per_deg_lon = 111320.0 * abs(math.cos(math.radians(center_lat)))

    delta_lon = half_side_m / meters_per_deg_lon
    delta_lat = half_side_m / meters_per_deg_lat

    return ee.Geometry.Rectangle([
        center_lon - delta_lon,
        center_lat - delta_lat,
        center_lon + delta_lon,
        center_lat + delta_lat,
    ])


def _safe_id(value, fallback):
    """Turn a footprint name / filename into something usable in an EE task name."""
    if isinstance(value, str) and value.strip():
        cleaned = re.sub(r"[^A-Za-z0-9_-]+", "-", value.strip()).strip("-")
        if cleaned:
            return cleaned[:40]
    return fallback


def iter_footprint_rois(csv_path, size_m=170):
    """Yield (footprint_id, roi) for EVERY footprint (row) in a CSV.

    Each footprint gets its OWN size_m x size_m box centered on that footprint's
    centroid. This is the per-footprint behaviour the pipeline needs -- do NOT
    average all footprints into a single box.
    """
    df = pd.read_csv(csv_path, low_memory=False)
    if ".geo" not in df.columns:
        raise ValueError(f"{csv_path} has no '.geo' column.")

    for idx, row in df.iterrows():
        geom_text = row.get(".geo")
        if not isinstance(geom_text, str) or not geom_text.strip():
            continue
        try:
            geom = json.loads(geom_text)
        except json.JSONDecodeError:
            continue

        centroid = centroid_from_geometry(geom)
        if not centroid:
            continue

        center_lon, center_lat = centroid
        roi = rectangle_around(center_lon, center_lat, size_m=size_m)
        footprint_id = _safe_id(row.get("name"), f"f{idx:04d}")
        yield footprint_id, roi


def roi_from_csv(csv_path, size_m=170):
    """DEPRECATED for the export pipeline: returns ONE box around the mean
    center of all footprints. Kept only for the single-AOI SAR/S2 examples.
    For the real per-footprint export use iter_footprint_rois()."""
    df = pd.read_csv(csv_path, low_memory=False)

    centroids = []
    for geom_text in df.get(".geo", []):
        if not isinstance(geom_text, str) or not geom_text.strip():
            continue
        try:
            geom = json.loads(geom_text)
        except json.JSONDecodeError:
            continue
        centroid = centroid_from_geometry(geom)
        if centroid:
            centroids.append(centroid)

    if not centroids:
        raise ValueError("No valid geometries found in the CSV.")

    center_lon = sum(lon for lon, _ in centroids) / len(centroids)
    center_lat = sum(lat for _, lat in centroids) / len(centroids)
    return rectangle_around(center_lon, center_lat, size_m=size_m)

In [9]:
def export_sar_to_drive(aoi, start_date, end_date, drive_folder="sar_exports", description="sar_export", scale=10):
    """Export one SAR image from Earth Engine to Google Drive."""
    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .select(["VV", "VH"])
        .sort("system:time_start")
    )
    image = collection.first()
    if image is None:
        raise ValueError("No SAR images matched the requested AOI and date range.")
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=drive_folder,
        fileNamePrefix=description,
        region=aoi,
        scale=scale,
        crs="EPSG:4326",
        fileFormat="GeoTIFF",
    )
    task.start()
    print(f"SAR task started: {description}")

In [17]:
import glob

# S2 reflectance bands are all uint16 and safe to export together.
# (The earlier FAILED S2 tasks came from exporting ALL S2 bands at once,
# which mixes uint16 reflectance with uint8 QA/metadata bands.)
S2_BANDS = ["B2", "B3", "B4", "B8"]   # blue, green, red, NIR
SAR_BANDS = ["VV", "VH"]


def export_s2_cloudfree_image(aoi, start_date, end_date, max_cloud_cover=10, bands=S2_BANDS):
    """Return the least-cloudy Sentinel-2 image in a date range, or None if none exist."""
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", max_cloud_cover))
        .sort("CLOUDY_PIXEL_PERCENTAGE")
        .select(bands)
    )
    if collection.size().getInfo() == 0:      # .first() is never Python None, so check size
        return None
    return collection.first()


def export_s2_cloudfree_to_drive(aoi, start_date, end_date, drive_folder="s2_exports",
                                 description="s2_export", scale=10, max_cloud_cover=10,
                                 bands=S2_BANDS):
    """Export a single cloud-free Sentinel-2 image (used by the simple example)."""
    image = export_s2_cloudfree_image(aoi, start_date, end_date,
                                       max_cloud_cover=max_cloud_cover, bands=bands)
    if image is None:
        raise ValueError("No cloud-free Sentinel-2 images matched the request.")
    task = ee.batch.Export.image.toDrive(
        image=image, description=description, folder=drive_folder,
        fileNamePrefix=description, region=aoi, scale=scale,
        crs="EPSG:4326", fileFormat="GeoTIFF",
    )
    task.start()
    print(f"S2 task started: {description}")


def _sar_mean(aoi, start_date, end_date, bands=SAR_BANDS):
    """Mean SAR image over a period, or None if no scenes intersect the AOI/date range."""
    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .select(bands)
    )
    if collection.size().getInfo() == 0:
        return None
    return collection.mean()


def export_siamese_pair_to_drive(csv_path, city, war_start, inference_start,
                                 size_m=170, drive_folder="siamese_exports",
                                 scale=10, max_cloud_cover=10,
                                 limit=None, start_index=0, dry_run=False):
    """Export before/after SAR + Sentinel-2 pairs for EVERY footprint in ONE CSV.

    One size_m box PER footprint (not one averaged box for the whole file).
      before = [war_start - 12 months, war_start]
      after  = [inference_start, inference_start + 1 month]
    Returns the list of started ee.batch.Task objects.
    """
    war_start_dt = ee.Date(war_start)
    before_start = war_start_dt.advance(-12, "month")
    before_end = war_start_dt
    after_start = ee.Date(inference_start)
    after_end = after_start.advance(1, "month")

    file_tag = _safe_id(os.path.splitext(os.path.basename(csv_path))[0], "file")

    footprints = list(iter_footprint_rois(csv_path, size_m=size_m))
    if start_index:
        footprints = footprints[start_index:]
    if limit is not None:
        footprints = footprints[:limit]

    print(f"{city}: {len(footprints)} footprint(s) from {os.path.basename(csv_path)}"
          + (" [dry run]" if dry_run else ""))

    tasks = []

    def start_export(image, description, aoi):
        task = ee.batch.Export.image.toDrive(
            image=image, description=description, folder=drive_folder,
            fileNamePrefix=description, region=aoi, scale=scale,
            crs="EPSG:4326", fileFormat="GeoTIFF",
        )
        task.start()
        tasks.append(task)
        print(f"  started: {description}")

    for footprint_id, aoi in footprints:
        base = f"{file_tag}_{footprint_id}"

        if dry_run:
            for suffix in ("sar_before", "sar_after", "s2_before", "s2_after"):
                print(f"  would export: {base}_{suffix}")
            continue

        try:
            images = [
                (_sar_mean(aoi, before_start, before_end), "sar_before"),
                (_sar_mean(aoi, after_start, after_end), "sar_after"),
                (export_s2_cloudfree_image(aoi, before_start, before_end,
                                           max_cloud_cover=max_cloud_cover), "s2_before"),
                (export_s2_cloudfree_image(aoi, after_start, after_end,
                                           max_cloud_cover=max_cloud_cover), "s2_after"),
            ]
            for image, suffix in images:
                if image is None:
                    print(f"  skip (no imagery): {base}_{suffix}")
                    continue
                start_export(image, f"{base}_{suffix}", aoi)
        except Exception as exc:
            print(f"  error on {base}: {exc}")

    return tasks


def export_city_footprints(city, data_dir="data/one_month", size_m=170,
                           drive_folder=None, scale=10, max_cloud_cover=10,
                           limit_per_file=None, dry_run=False):
    """Export before/after pairs for EVERY footprint in EVERY CSV of a city.

    This is 'the entirety of Gaza': it globs all {city}_*_footprints.csv files,
    reads the label/inference date from each FILENAME, uses CITY_CONFIG for the
    war-start date, and exports one size_m pair per footprint.
    """
    if city not in CITY_CONFIG:
        raise ValueError(f"No war_start configured for {city} in CITY_CONFIG.")
    war_start = CITY_CONFIG[city]

    if drive_folder is None:
        drive_folder = f"{city.lower()}_siamese_exports"

    pattern = os.path.join(data_dir, f"{city}_*_footprints.csv")
    csv_paths = sorted(glob.glob(pattern))
    if not csv_paths:
        raise FileNotFoundError(f"No CSVs matched {pattern}")

    print(f"{city}: {len(csv_paths)} CSV file(s) matched {pattern}\n")

    all_tasks = []
    for csv_path in csv_paths:
        _, inference_start = parse_name(csv_path)   # label date comes from the filename
        tasks = export_siamese_pair_to_drive(
            csv_path=csv_path, city=city, war_start=war_start,
            inference_start=inference_start, size_m=size_m,
            drive_folder=drive_folder, scale=scale,
            max_cloud_cover=max_cloud_cover, limit=limit_per_file,
            dry_run=dry_run,
        )
        all_tasks.extend(tasks)

    print(f"\n{city}: {len(all_tasks)} export task(s) started across "
          f"{len(csv_paths)} file(s), folder='{drive_folder}'.")
    return all_tasks

In [23]:
import numpy as np, rasterio
from rasterio.windows import Window
from rasterio.warp import transform as warp_transform

import math, glob, os

import glob, math

def footprint_bbox(csv_path, pad_m=200):
    """Bounding rectangle covering all footprints in a CSV — only used to prune
    the image collections. One light pass over the .geo column."""
    df = pd.read_csv(csv_path, low_memory=False, usecols=[".geo"])
    lons, lats = [], []
    for geom_text in df[".geo"]:
        if not isinstance(geom_text, str) or not geom_text.strip():
            continue
        try:
            c = centroid_from_geometry(json.loads(geom_text))
        except json.JSONDecodeError:
            continue
        if c:
            lons.append(c[0]); lats.append(c[1])
    if not lons:
        raise ValueError(f"No geometries in {csv_path}")
    mean_lat = sum(lats) / len(lats)
    pad_lat = pad_m / 111320.0
    pad_lon = pad_m / (111320.0 * abs(math.cos(math.radians(mean_lat))))
    return ee.Geometry.Rectangle([min(lons) - pad_lon, min(lats) - pad_lat,
                                  max(lons) + pad_lon, max(lats) + pad_lat])

def _sar_composite(start, end, region):
    return (ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterBounds(region).filterDate(start, end)
            .select(SAR_BANDS).mean())

def _s2_composite(start, end, region, max_cloud_cover=10):
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterBounds(region).filterDate(start, end)
           .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", max_cloud_cover)))
    def mask(img):
        scl = img.select("SCL")            # drop shadow/cloud/cirrus/snow classes
        clear = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
                    .And(scl.neq(10)).And(scl.neq(11)))
        return img.updateMask(clear)
    return col.map(mask).select(S2_BANDS).median()

def utm_epsg(lon, lat):
    zone = int((lon + 180) // 6) + 1
    return (32600 if lat >= 0 else 32700) + zone

def city_aoi_and_crs(csv_paths, pad_m=500):
    """Union bbox over all a city's footprints + its UTM EPSG code."""
    lons, lats = [], []
    for p in csv_paths:
        for t in pd.read_csv(p, low_memory=False, usecols=[".geo"])[".geo"]:
            if isinstance(t, str) and t.strip():
                try: c = centroid_from_geometry(json.loads(t))
                except json.JSONDecodeError: continue
                if c: lons.append(c[0]); lats.append(c[1])
    if not lons: raise ValueError("no footprints found")
    mlat = sum(lats) / len(lats)
    dlat = pad_m / 111320.0
    dlon = pad_m / (111320.0 * abs(math.cos(math.radians(mlat))))
    bbox = [min(lons)-dlon, min(lats)-dlat, max(lons)+dlon, max(lats)+dlat]
    return bbox, utm_epsg(sum(lons)/len(lons), mlat)

def export_city_mosaics(city, data_dir="data/one_month", scale=10,
                        max_cloud_cover=10, drive_folder=None, dry_run=False):
    csv_paths = sorted(glob.glob(os.path.join(data_dir, f"{city}_*_footprints.csv")))
    if not csv_paths: raise FileNotFoundError(city)
    war = ee.Date(CITY_CONFIG[city])
    before_start, before_end = war.advance(-12, "month"), war
    bbox, epsg = city_aoi_and_crs(csv_paths)
    region, crs = ee.Geometry.Rectangle(bbox), f"EPSG:{epsg}"
    folder = drive_folder or f"{city.lower()}_mosaics"

    def export(img, desc):
        if dry_run: print("  would export", desc, "in", crs); return
        ee.batch.Export.image.toDrive(
            image=img, description=desc, folder=folder, fileNamePrefix=desc,
            region=region, scale=scale, crs=crs, maxPixels=int(1e10),
            fileFormat="GeoTIFF").start()
        print("  started", desc)

    # "before" depends only on war_start -> one pair per city
    export(_sar_composite(before_start, before_end, region), f"{city}_sar_before")
    export(_s2_composite(before_start, before_end, region, max_cloud_cover), f"{city}_s2_before")
    # "after" depends on each file's label date
    for p in csv_paths:
        _, inf = parse_name(p)
        a0 = ee.Date(inf); a1 = a0.advance(1, "month")
        tag = _safe_id(os.path.splitext(os.path.basename(p))[0], "file")
        export(_sar_composite(a0, a1, region), f"{tag}_sar_after")
        export(_s2_composite(a0, a1, region, max_cloud_cover), f"{tag}_s2_after")

# for city in CITY_CONFIG: export_city_mosaics(city)   # ~100-150 tasks for everything

def layer_source(mosaic_dir, layer_name):
    """Return a path to open. If EE sharded the export, VRT the pieces together."""
    tifs = sorted(glob.glob(os.path.join(mosaic_dir, f"{layer_name}*.tif")))
    if not tifs: raise FileNotFoundError(layer_name)
    if len(tifs) == 1: return tifs[0]
    vrt = os.path.join(mosaic_dir, f"{layer_name}.vrt")
    os.system(f"gdalbuildvrt -q {vrt} " + " ".join(tifs))   # seamless mosaic
    return vrt

def crop_layer_to_array(csv_path, source, size_m=170, id_col="name"):
    with rasterio.open(source) as src:
        npx = round(size_m / src.res[0]); off = npx // 2
        df = pd.read_csv(csv_path, low_memory=False)

        rows = []
        for idx, row in df.iterrows():
            t = row.get(".geo")
            if not isinstance(t, str) or not t.strip(): continue
            try: c = centroid_from_geometry(json.loads(t))
            except json.JSONDecodeError: continue
            if c: rows.append((idx, row, c))

        # transform ALL centroids at once (lon/lat -> raster CRS)
        xs, ys = warp_transform("EPSG:4326", src.crs,
                                [c[0] for *_, c in rows], [c[1] for *_, c in rows])
        ids, chips = [], []
        for (idx, row, _), x, y in zip(rows, xs, ys):
            r, col = src.index(x, y)
            w = Window(col - off, r - off, npx, npx)
            chips.append(src.read(window=w, boundless=True, fill_value=0))
            ids.append(_safe_id(row.get(id_col), f"f{idx:04d}"))
    return np.array(ids), np.stack(chips)   # chips: (N, bands, npx, npx)

# Example: one paired output file per (csv, layer)
def crop_city(city, csv_path, mosaic_dir, out_dir="chips"):
    os.makedirs(out_dir, exist_ok=True)
    tag = _safe_id(os.path.splitext(os.path.basename(csv_path))[0], "file")
    layers = {"sar_before": f"{city}_sar_before", "s2_before": f"{city}_s2_before",
              "sar_after": f"{tag}_sar_after",     "s2_after": f"{tag}_s2_after"}
    for key, layer in layers.items():
        out = os.path.join(out_dir, f"{tag}_{key}.npz")
        if os.path.exists(out): continue          # resume: skip finished layers
        ids, arr = crop_layer_to_array(csv_path, layer_source(mosaic_dir, layer))
        np.savez_compressed(out, ids=ids, chips=arr)
        print(f"  {out}: {arr.shape}")

## Example usage

The cells below are intentionally left as examples. Change the AOI, dates, and folder names before running them.

In [25]:
# ============================================================
# PHASE 1 — export whole-city mosaics to Google Drive
# ============================================================

# 1a. Dry run first — prints what it would export and in which CRS, no tasks started.
#export_city_mosaics("Gaza", dry_run=True)

# 1b. Real export for one city (a handful of tasks -> My Drive/gaza_mosaics).
export_city_mosaics("Gaza")

# 1c. Or every configured city at once (~100-150 tasks total, submitted in seconds).
#for city in CITY_CONFIG:
#    export_city_mosaics(city)


# ------------------------------------------------------------
# Optional: block until the exports finish before cropping.
# ------------------------------------------------------------
import time

def wait_for_tasks(name_prefix=None, poll=60):
    """Poll until all matching EXPORT_IMAGE tasks leave the running/pending state."""
    active = {"READY", "RUNNING"}
    while True:
        states = []
        for t in ee.batch.Task.list():
            s = t.status()
            if s.get("task_type") != "EXPORT_IMAGE":
                continue
            if name_prefix and not s.get("description", "").startswith(name_prefix):
                continue
            states.append(s.get("state"))
        pending = [x for x in states if x in active]
        print(f"{len(pending)} running/pending | "
              f"{states.count('COMPLETED')} completed | {states.count('FAILED')} failed")
        if not pending:
            return
        time.sleep(poll)

wait_for_tasks("Gaza")   # returns once Gaza's mosaics are done

  started Gaza_sar_before
  started Gaza_s2_before
  started Gaza_20240503_1_footprints_sar_after
  started Gaza_20240503_1_footprints_s2_after
4 running/pending | 2 completed | 2 failed
4 running/pending | 2 completed | 2 failed
4 running/pending | 2 completed | 2 failed
4 running/pending | 2 completed | 2 failed
4 running/pending | 2 completed | 2 failed
4 running/pending | 2 completed | 2 failed
4 running/pending | 2 completed | 2 failed
3 running/pending | 3 completed | 2 failed
3 running/pending | 3 completed | 2 failed
3 running/pending | 3 completed | 2 failed
3 running/pending | 3 completed | 2 failed
3 running/pending | 3 completed | 2 failed
3 running/pending | 3 completed | 2 failed
2 running/pending | 4 completed | 2 failed
1 running/pending | 5 completed | 2 failed
1 running/pending | 5 completed | 2 failed
0 running/pending | 6 completed | 2 failed


In [ ]:
# ------------------------------------------------------------
# Load a cropped result and pair before/after by footprint id.
# ------------------------------------------------------------
# import numpy as np

# tag = "Gaza_20240503_1_footprints"
# s2_before = np.load(f"chips/{tag}_s2_before.npz", allow_pickle=True)
# s2_after  = np.load(f"chips/{tag}_s2_after.npz",  allow_pickle=True)

# # ids line up row-for-row (same CSV, same order), so pairing is just:
# ids   = s2_before["ids"]                 # (N,)  footprint identifiers
# before = s2_before["chips"]              # (N, 4, 17, 17)  B2,B3,B4,B8
# after  = s2_after["chips"]               # (N, 4, 17, 17)
# assert np.array_equal(ids, s2_after["ids"])
# print(before.shape, after.shape, "->", len(ids), "paired footprints")

In [ ]:
### COLAB TASKS #####
!pip install -q rasterio
!apt-get -qq install -y gdal-bin        

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob, os, shutil

def stage_local(drive_dir, layer_name, local_dir="/content/mosaics"):
    """Copy a layer's shard(s) from the Drive mount to local disk; return a
    readable path (a VRT if Earth Engine sharded the export)."""
    os.makedirs(local_dir, exist_ok=True)
    shards = sorted(glob.glob(os.path.join(drive_dir, f"{layer_name}*.tif")))
    if not shards:
        raise FileNotFoundError(f"{layer_name} not found in {drive_dir}")
    local = []
    for s in shards:
        dst = os.path.join(local_dir, os.path.basename(s))
        if not os.path.exists(dst):
            shutil.copy(s, dst)         # sequential read off the mount = fine
        local.append(dst)
    if len(local) == 1:
        return local[0]
    vrt = os.path.join(local_dir, f"{layer_name}.vrt")
    os.system(f"gdalbuildvrt -q {vrt} " + " ".join(local))
    return vrt

import numpy as np

DRIVE_ROOT = "/content/drive/MyDrive"

def crop_city_colab(city, csv_path, out_dir=None):
    mosaic_dir = f"{DRIVE_ROOT}/{city.lower()}_mosaics"       # Phase 1 output on Drive
    out_dir = out_dir or f"{DRIVE_ROOT}/{city.lower()}_chips" # chips written back to Drive
    os.makedirs(out_dir, exist_ok=True)
    tag = _safe_id(os.path.splitext(os.path.basename(csv_path))[0], "file")
    layers = {"sar_before": f"{city}_sar_before", "s2_before": f"{city}_s2_before",
              "sar_after":  f"{tag}_sar_after",    "s2_after":  f"{tag}_s2_after"}
    for key, layer in layers.items():
        out = os.path.join(out_dir, f"{tag}_{key}.npz")
        if os.path.exists(out):           # resume: skip finished layers
            continue
        source = stage_local(mosaic_dir, layer)   # Drive -> local, returns path or VRT
        ids, arr = crop_layer_to_array(csv_path, source)
        np.savez_compressed(out, ids=ids, chips=arr)
        print(f"  {out}: {arr.shape}")

In [ ]:
import glob

city, data_dir = "Gaza", "data/one_month"
# NOTE: the footprint CSVs must also be reachable in Colab — keep them in Drive
# (e.g. data_dir = f"{DRIVE_ROOT}/footprints/one_month") or upload them.

for csv_path in sorted(glob.glob(f"{data_dir}/{city}_*_footprints.csv")):
    crop_city_colab(city, csv_path)
# -> writes to  My Drive/gaza_chips/Gaza_20240503_1_footprints_{sar,s2}_{before,after}.npz